# 01 · Training the Score Network

*Code companion to an MSc dissertation on score-based generative
modelling of financial time series (Queen Mary University of London,
2026) — see the repository README for the reference.*

This notebook trains the score network on windowed S&P 500 daily log returns
by denoising score matching under the Variance-Exploding (VE) SDE
(Song et al., 2021).

**Pipeline**

1. Download the price series and compute daily log returns.
2. Build the sliding-window training set (window length `L = 256`, stride `s`), standardised to zero mean and unit variance. Strides 1, 10 and 20 give the datasets D1 (17,224 windows), D2 (1,723) and D3 (862).
3. Instantiate the score network from `config.model_config`.
4. Train with denoising score matching and save a checkpoint.
5. Plot the training loss curve.

**Outputs**

| Artefact | Location |
|---|---|
| Model checkpoint (weights, optimiser, configs, loss history) | `checkpoints/<run_name>.pt` |
| Loss curve | `figures/<run_name>_loss.png` |

All hyperparameters live in `config.py`. Change them there rather than in this notebook so that every run is reproducible from a single source of truth.

## 0 · Setup

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import torch

from config import data_config, model_config
from data import get_data, DailyLogReturnsData
from model import ScoreNet, train_net
from plots import plot_loss_curve

## 1 · Data

The underlying series is the S&P 500 index (`^GSPC`) downloaded via `yfinance`
over the date range set in `config.data_config`. Daily log returns are computed
from the close price.

In [ ]:
snp_data = get_data({
    "ticker": data_config["TICKER"],
    "start":  data_config["START"],
    "end":    data_config["END"],
})
log_returns = snp_data["log_return"]

print(f"Ticker:            {data_config['TICKER']}")
print(f"Date range:        {log_returns.index.min().date()} to {log_returns.index.max().date()}")
print(f"Log-return series: {len(log_returns):,} observations")

### 1.1 · Sliding-window training set

Windows of length `WINDOW_SIZE` are extracted with step `STRIDE`. Smaller
strides yield more (but more overlapping) windows; see the README for the
dataset variants used in the dissertation. Returns are standardised so the
model operates on a unit-variance scale.

In [ ]:
dataset = DailyLogReturnsData(
    log_returns=log_returns,
    window_size=data_config["WINDOW_SIZE"],
    stride=data_config["STRIDE"],
    normalize=True,
)
X_train = torch.stack([dataset[i].squeeze(0) for i in range(len(dataset))])

print(f"Training set:   {tuple(X_train.shape)}  (windows, window length)")
print(f"Stride:         {data_config['STRIDE']}")
print(f"Pooled std:     {X_train.std():.4f}  (≈ 1 after standardisation)")
print(f"Raw mean / std: {dataset.mean:.6f} / {dataset.std:.6f}  (kept for de-standardisation)")

## 2 · Model

The score network is a residual convolutional backbone with a temporal
transformer layer in each block (WaveNet/DiffWave residual stack with the
CSDI temporal transformer, Tashiro et al., 2021). The three configurations differ only in the number of convolutional channels and
residual blocks: Conf. 1 (64, 1), Conf. 2 (64, 2), Conf. 3 (128, 2).
See `model/` for the implementation.

In [ ]:
net = ScoreNet(
    channels      = model_config["CHANNELS"],
    diffusion_dim = model_config["DIFFUSION_DIM"],
    num_heads     = model_config["NUM_HEADS"],
    num_blocks    = model_config["NUM_RES_BLOCKS"],
    kernel_size   = model_config["KERNEL_SIZE"],
)

n_params = sum(p.numel() for p in net.parameters() if p.requires_grad)
print(f"Configuration: {model_config['CHANNELS']} channels × {model_config['NUM_RES_BLOCKS']} residual blocks")
print(f"Trainable parameters: {n_params:,}")
print(f"Device: {model_config['DEVICE']}")

## 3 · Training

Denoising score matching under the VE-SDE with the exponential noise schedule
$\sigma(t) = \sigma_{\min}(\sigma_{\max}/\sigma_{\min})^t$, $t \sim \mathcal{U}(0,1)$,
$\sigma_{\min}=0.01$, $\sigma_{\max}=10$, and the $\sigma^2$-weighted loss
$\mathbb{E}\,\|\sigma(t)\, s_\theta(x_t, t) + \varepsilon\|^2$.

The run name encodes the configuration and dataset so that checkpoints and
generated samples remain traceable: `conf<k>_s<stride>_noise<sigma_max>_seed<seed>`.

In [ ]:
RUN_NAME = f"conf{model_config['NUM_RES_BLOCKS']}_s{data_config['STRIDE']}_noise10_seed0"

train_config = {
    "iter":       1,                   # epochs
    "batch_size": 64,
    "lr":         1e-3,
    "drop_lr":    80,                    # epoch at which lr is divided by 10
    "log_every":  10,
    "in_feat":    X_train.shape[1],
    "out_feat":   X_train.shape[1],
    "save_path":  f"./checkpoints/{RUN_NAME}.pt",
    "device":     model_config["DEVICE"],
}

losses = train_net(X_train, net, train_config, model_config, seed=0)

## 4 · Loss curve

In [ ]:
plot_loss_curve(losses, save_path=f"./figures/{RUN_NAME}_loss.png",
                title=f"Training loss — {RUN_NAME}")

## 5 · Verify the checkpoint

Reload the saved checkpoint to confirm that weights and configurations
round-trip correctly. The `sampling` and `evaluation` notebooks rebuild the
network from the configuration stored inside the checkpoint, so nothing else
needs to be recorded manually.

In [ ]:
checkpoint = torch.load(train_config["save_path"], map_location="cpu", weights_only=False)

net_reloaded = ScoreNet(
    channels      = checkpoint["model_config"]["CHANNELS"],
    diffusion_dim = checkpoint["model_config"]["DIFFUSION_DIM"],
    num_heads     = checkpoint["model_config"]["NUM_HEADS"],
    num_blocks    = checkpoint["model_config"]["NUM_RES_BLOCKS"],
    kernel_size   = checkpoint["model_config"]["KERNEL_SIZE"],
)
net_reloaded.load_state_dict(checkpoint["model_state_dict"])

print(f"Checkpoint:  {train_config['save_path']}")
print(f"Epochs run:  {checkpoint['epoch']}")
print(f"Final loss:  {checkpoint['losses'][-1]:.5f}")
print(f"Model conf:  {checkpoint['model_config']}")